# AttnForge ALiBi CUDA Extension Test

Build and validate the AttnForge ALiBi CUDA extension against its PyTorch reference implementation. Run the sections in order in a GPU-enabled Google Colab runtime.

<a href="https://colab.research.google.com/github/SigwiLi/AttnForge/blob/main/AttnForgeTest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Check the GPU environment

Display the NVIDIA driver, CUDA compiler, PyTorch version, CUDA availability, and active GPU.

In [1]:
!nvidia-smi
!nvcc --version

import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("PyTorch CUDA:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Mon Oct 12 15:58:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   76C    P8             22W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install the Ninja build system

Install Ninja, which PyTorch uses to compile the CUDA extension, and print its version.

In [2]:
!pip install -q ninja
!ninja --version

1.13.2.git.kitware.jobserver-pipe-1
Traceback (most recent call last):
  File "/usr/local/bin/reference", line 5, in <module>
    from reference.reference import main
  File "/usr/local/lib/python3.13/dist-packages/reference/reference.py", line 27
    print line.rstrip()
    ^^^^^^^^^^^^^^^^^^^
SyntaxError: Missing parentheses in call to 'print'. Did you mean print(...)?


## 3. Clone or update AttnForge

Clone the repository into Colab, or safely update an existing checkout, then confirm that the CUDA and reference source files are present.

In [3]:
import os
import subprocess
from pathlib import Path

repo_url = "https://github.com/SigwiLi/AttnForge.git"
repo_path = Path("/content/AttnForge")


def run_git(arguments):
    result = subprocess.run(
        ["git", *arguments],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    print(result.stdout)

    if result.returncode != 0:
        raise RuntimeError(
            f"Git command failed: git {' '.join(arguments)}"
        )


if not (repo_path / ".git").exists():
    print("Cloning AttnForge...")

    run_git([
        "clone",
        "-b",
        "main",
        repo_url,
        str(repo_path),
    ])

else:
    print("AttnForge already exists.")

    status = subprocess.run(
        [
            "git",
            "-C",
            str(repo_path),
            "status",
            "--porcelain",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=True,
    )

    if status.stdout.strip():
        print("Stashing temporary Colab changes...")

        run_git([
            "-C",
            str(repo_path),
            "stash",
            "push",
            "-u",
            "-m",
            "Temporary Colab changes before pull",
        ])

    print("Pulling the latest main branch...")

    run_git([
        "-C",
        str(repo_path),
        "pull",
        "--rebase",
        "origin",
        "main",
    ])


os.chdir(repo_path)

print("Current directory:", os.getcwd())

run_git([
    "-C",
    str(repo_path),
    "status",
    "--short",
    "--branch",
])

cuda_source = repo_path / "kernels" / "alibi_kernel.cu"
reference_source = repo_path / "reference.py"

print("CUDA source exists:", cuda_source.exists())
print("Reference exists:", reference_source.exists())

AttnForge already exists.
Pulling the latest main branch...
From https://github.com/SigwiLi/AttnForge
 * branch            main       -> FETCH_HEAD
Already up to date.

Current directory: /content/AttnForge
## main...origin/main

CUDA source exists: True
Reference exists: True


## 4. Validate the CUDA source

Check that the CUDA source contains the kernel, launcher, PyTorch wrapper, and Python binding required by the extension.

In [5]:
from pathlib import Path

source_path = Path(
    "/content/AttnForge/kernels/alibi_kernel.cu"
)

assert source_path.exists(), (
    f"CUDA source does not exist: {source_path}"
)

source_text = source_path.read_text()

required_parts = [
    "alibi_forward_kernel",
    "launch_alibi_forward",
    "torch::Tensor alibi_forward",
    "PYBIND11_MODULE",
]

missing_parts = [
    name
    for name in required_parts
    if name not in source_text
]

assert not missing_parts, (
    f"CUDA extension is missing: {missing_parts}"
)

print("CUDA kernel, launcher, wrapper, and binding found.")

CUDA kernel, launcher, wrapper, and binding found.


## 5. Clear the extension cache

Remove any stale or failed PyTorch extension build so the next compilation starts cleanly.

In [6]:
from pathlib import Path
import shutil
from torch.utils.cpp_extension import _get_build_directory

build_directory = Path(
    _get_build_directory(
        "attnforge_alibi_v2",
        verbose=False,
    )
)

print("Removing failed cache:", build_directory)

shutil.rmtree(
    build_directory,
    ignore_errors=True,
)

Removing failed cache: /root/.cache/torch_extensions/py313_cu130/attnforge_alibi_v2


## 6. Build and load the CUDA extension

Compile the ALiBi CUDA source with optimization enabled and load the resulting module into Python.

In [7]:
import os
from pathlib import Path
from torch.utils.cpp_extension import load

os.environ["MAX_JOBS"] = "1"

source_path = Path(
    "/content/AttnForge/kernels/alibi_kernel.cu"
)

build_path = Path(
    "/content/attnforge_build_v3"
)

build_path.mkdir(
    parents=True,
    exist_ok=True,
)

print("Source exists:", source_path.exists())
print("Build directory:", build_path)

alibi_cuda = load(
    name="attnforge_alibi_v3",
    sources=[str(source_path)],
    build_directory=str(build_path),
    extra_cflags=["-O3"],
    extra_cuda_cflags=["-O3"],
    with_cuda=True,
    verbose=True,
)

print("Extension loaded:", hasattr(alibi_cuda, "forward"))

Source exists: True
Build directory: /content/attnforge_build_v3
Extension loaded: True


## 7. Load the PyTorch reference implementation

Import `alibi_reference` directly from the repository for use as the correctness baseline.

In [10]:
import importlib.util
from pathlib import Path

reference_path = Path(
    "/content/AttnForge/reference.py"
)

assert reference_path.exists()

spec = importlib.util.spec_from_file_location(
    "attnforge_reference",
    reference_path,
)

attnforge_reference = importlib.util.module_from_spec(spec)
spec.loader.exec_module(attnforge_reference)

alibi_reference = attnforge_reference.alibi_reference

print("Loaded reference from:", reference_path)

Loaded reference from: /content/AttnForge/reference.py


## 8. Compare CUDA output with the reference

Create a deterministic test input, run both implementations, and assert that their outputs agree within floating-point tolerances.

In [11]:
import sys
import torch

sys.path.insert(0, "/content/AttnForge")

from reference import alibi_reference

torch.manual_seed(42)

B = 2
H = 3
N = 4

scores = torch.randn(
    B,
    H,
    N,
    N,
    device="cuda",
    dtype=torch.float32,
)

slopes = torch.tensor(
    [0.5, 0.25, 0.125],
    device="cuda",
    dtype=torch.float32,
)

expected = alibi_reference(scores, slopes)

actual = alibi_cuda.forward(
    scores,
    slopes,
)

torch.cuda.synchronize()

torch.testing.assert_close(
    actual,
    expected,
    rtol=1e-5,
    atol=1e-6,
)

print("CUDA output matches the PyTorch reference.")

print("\nPyTorch reference:")
print(expected[0, 0])

print("\nCUDA output:")
print(actual[0, 0])

CUDA output matches the PyTorch reference.

PyTorch reference:
tensor([[ 0.1940,    -inf,    -inf,    -inf],
        [-2.4244,  0.6530,    -inf,    -inf],
        [-0.4720, -1.7753, -1.6621,    -inf],
        [-1.5926, -0.8008, -1.6204,  1.8577]], device='cuda:0')

CUDA output:
tensor([[ 0.1940,    -inf,    -inf,    -inf],
        [-2.4244,  0.6530,    -inf,    -inf],
        [-0.4720, -1.7753, -1.6621,    -inf],
        [-1.5926, -0.8008, -1.6204,  1.8577]], device='cuda:0')


## 9. Inspect normalized attention weights

Apply softmax to the CUDA result, display one attention matrix, and verify that each row sums to one.

In [12]:
weights = torch.softmax(actual, dim=-1)

print("Attention weights:")
print(weights[0, 0])

print("\nRow sums:")
print(weights[0, 0].sum(dim=-1))

Attention weights:
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.0440, 0.9560, 0.0000, 0.0000],
        [0.6346, 0.1724, 0.1930, 0.0000],
        [0.0280, 0.0619, 0.0273, 0.8829]], device='cuda:0')

Row sums:
tensor([1.0000, 1.0000, 1.0000, 1.0000], device='cuda:0')
